# Video 04 - LLMs have no memory

**You are here:** Video 4 of 13 (see `../you-are-here.svg`). On the agent anatomy picture, today lights up **Memory**.

**The one idea:** the model is **stateless**. **Memory is the list you send.**

**By the end you can:**
1. Explain why an LLM forgets everything between calls.
2. Build a chat that remembers.
3. Explain why memory makes every call bigger and pricier.
4. Trim history to save tokens - and pin facts so they are not lost.

**Before you start:** attach this notebook to **Serverless**.

In [0]:
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"
print("Connected. Using model:", MODEL)

## Part 1 - Proof that the model forgets
Two **separate** calls. Nothing connects them (diagram: `../diagrams/01-stateless.svg`).

**Predict first:** will the second call know the name?

In [0]:
first = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "Hi, my name is Sam."}]
)
print("Call 1:", first.choices[0].message.content)

second = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "What is my name?"}]
)
print("Call 2:", second.choices[0].message.content)

In [0]:
# The fix: send the earlier messages along with the new question
history = [
    {"role": "user", "content": "Hi, my name is Sam."},
    {"role": "assistant", "content": first.choices[0].message.content},   # the model's earlier reply
    {"role": "user", "content": "What is my name?"},
]
response = client.chat.completions.create(model=MODEL, messages=history)
print(response.choices[0].message.content)

## Part 2 - A chat that remembers
`Chat` keeps the list for us. `send()` does three things:
1. add the user's message to the list,
2. call the model with the **whole** list,
3. add the model's reply to the list.

It also records the input tokens of every call so we can watch the cost.

In [0]:
class Chat:
    def __init__(self, system="You are a friendly, concise assistant. Keep answers under 40 words.", max_history=None):
        self.system = {"role": "system", "content": system}
        self.history = []               # <- this list IS the memory
        self.max_history = max_history  # None = keep everything; a number = keep only the last N messages
        self.input_tokens = []          # input tokens of each call (for the cost meter)

    def send(self, text):
        self.history.append({"role": "user", "content": text})                   # 1. remember the user
        recent = self.history if self.max_history is None else self.history[-self.max_history:]
        response = client.chat.completions.create(model=MODEL, messages=[self.system] + recent)  # 2. send it all
        reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": reply})            # 3. remember the reply
        self.input_tokens.append(response.usage.prompt_tokens)
        return reply

In [0]:
TURNS = [
    "Hi, my name is Sam and I live in Perth.",
    "Suggest a fun weekend activity.",
    "What should I pack for it?",
    "Any food recommendations nearby?",
    "How long would the whole day take?",
    "What is my name and where do I live?",   # the memory test
]

chat = Chat()
for turn in TURNS:
    print(f"You : {turn}")
    print(f"Bot : {chat.send(turn)}\n")

In [0]:
# The memory is just a Python list. You can read it, save it - even edit it.
print(f"{len(chat.history)} messages in memory:\n")
for i, m in enumerate(chat.history):
    print(f"{i:2}. [{m['role']:9}] {m['content'][:70]}")

## Cost meter: memory isn't free
Every call resends the **whole** history (diagram: `../diagrams/02-memory-is-a-list.svg`). So input tokens grow every turn - and you pay for the early messages again on every turn.

**Predict first:** how many times bigger is turn 6's input than turn 1's?

In [0]:
for turn_number, tokens in enumerate(chat.input_tokens, start=1):
    print(f"Turn {turn_number}: {tokens:4} input tokens  " + "#" * (tokens // 10))

print(f"\nTurn 6 is {chat.input_tokens[-1] / chat.input_tokens[0]:.1f}x turn 1")
print(f"Total input tokens for the whole conversation: {sum(chat.input_tokens)}")
# Every model also has a maximum input size - the CONTEXT WINDOW. Exceed it and the call fails.

## Part 3 - Trimming the history
Simplest fix for growing cost: only send the **last N messages**. Let's replay the same conversation keeping just the last 4.

**Predict first:** will it still know the name at the end?

In [0]:
trimmed = Chat(max_history=4)
for turn in TURNS:
    reply = trimmed.send(turn)

print("Input tokens per turn, full   :", chat.input_tokens)
print("Input tokens per turn, trimmed:", trimmed.input_tokens)
print(f"\nLast answer (trimmed): {reply}")

## What could go wrong? Trimming silently forgets
The name was in the **first** message - trimming threw it away, and the model doesn't warn you.

**Fix: pin** the important facts in the system prompt, which is always sent (diagram: `../diagrams/03-trimming-and-pinning.svg`).

In [0]:
pinned = Chat(
    system=(
        "You are a friendly, concise assistant. Keep answers under 40 words.\n"
        "Facts about the user: name is Sam; lives in Perth."   # <- pinned facts, always sent
    ),
    max_history=4,
)
for turn in TURNS:
    reply = pinned.send(turn)

print(f"Last answer (trimmed + pinned): {reply}")
print(f"Last call input tokens - full: {chat.input_tokens[-1]}, trimmed: {trimmed.input_tokens[-1]}, "
      f"trimmed + pinned: {pinned.input_tokens[-1]}")

## Why agents care
An agent's memory holds its plan, every tool result and every decision. What you keep in the list - and what you drop - decides how smart it can be. This is called **context engineering** (Video 07).

## Recap
* LLMs are **stateless**: each call starts from zero.
* **Memory is the list you send** - keep it yourself and send it every time.
* Memory makes every call bigger and pricier; the **context window** is the hard limit.
* **Trimming** saves tokens but can silently forget; **pin** what matters (or summarise).

## Practise
* [exercises.md](../exercises.md) - 3 exercises + bonus (about 15 minutes). Add your code below this cell.
* [quiz.md](../quiz.md) - 5 questions.
* Solutions: `04-no-memory-solutions` (try first!).

**Next:** Video 05 - Answers code can use: structured JSON output.